# Reply timing and the protocol

Two measurements: (1) reply counts and timing statistics of the compiled engine's
moves from the Cloudict fixtures' records; and (2) a protocol smoke test against
the compiled engine — identify, start as both colours, set depth, receive a move,
return a legal move, exit — with the transcript saved and startup, search and
end-to-end timing separated where measurable.

In [1]:
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
from pathlib import Path

sys.dont_write_bytecode = True  # never write __pycache__ into the existing project tree


def find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "connect6" / "defines.py").is_file():
            return base.resolve()
    raise RuntimeError("run this notebook from inside the Connect6Engine repository")


ROOT = find_root()
sys.path.insert(0, str(ROOT))

from connect6.board import init_board, make_move, check_game_end
from connect6.defines import Color, GameResult
from connect6.protocol import msg2move

HERE = ROOT / "benchmarks" / "assignment_coverage"
RECORDS = HERE / "records"
LOGS = HERE / "logs"
RECORDS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
OUT_JSON = RECORDS / "01_cloudict_positions.json"

OUR_BIN = ROOT / "dist" / "Connect6_Butovka_Ahmed"
assert OUR_BIN.is_file(), f"missing prerequisite, experiment blocked: {OUR_BIN}"

TARGET_S = 2.0   # the V6_TIME_BUDGET wall-budget target the engine is configured with
DEADLINE_S = 3.0  # the end-to-end deadline notebook 1 enforced per reply

if not OUT_JSON.is_file():
    print("BLOCKED: notebook 1's records are missing "
          f"({OUT_JSON}); run 01_cloudict_positions.ipynb first")
    records = None
else:
    records = json.loads(OUT_JSON.read_text())
    print(f"loaded notebook 1's records: {len(records['fixtures'])} fixtures "
          f"from {OUT_JSON.relative_to(ROOT)}")

loaded notebook 1's records: 14 fixtures from benchmarks/assignment_coverage/records/01_cloudict_positions.json


In [2]:
if records is None:
    print("reply-time analysis BLOCKED (see above)")
else:
    moves = [(f, m) for f in records["fixtures"] for m in f.get("moves", [])]
    ours = [m for _f, m in moves if m["engine"] == "ours"]
    theirs = [m for _f, m in moves if m["engine"] == "cloudict"]

    def stats_of(replies):
        walls = sorted(m["wall_s"] for m in replies)
        if not walls:
            return {"replies": 0}
        return {
            "replies": len(walls),
            "wall_mean_s": round(sum(walls) / len(walls), 3),
            "wall_median_s": walls[len(walls) // 2],
            "wall_max_s": walls[-1],
        }

    ours_stats, theirs_stats = stats_of(ours), stats_of(theirs)
    ours_searched = [m for m in ours if not m.get("book")]
    searched_stats = stats_of(ours_searched)
    book_replies = [m for m in ours if m.get("book")]
    target_misses = [m for m in ours if m["wall_s"] > TARGET_S]
    search_target_misses = [m for m in ours if "search_s" in m and m["search_s"] > TARGET_S]
    deadline_misses = [m for m in ours + theirs if m["wall_s"] > DEADLINE_S]
    deadline_fixtures = [f["id"] for f in records["fixtures"] if f["status"] == "timeout"]

    print(f"{'engine':<10} {'replies':>8} {'mean s':>7} {'median s':>8} {'max s':>7}")
    print(f"{'ours':<10} {ours_stats.get('replies', 0):>8} {str(ours_stats.get('wall_mean_s')):>7} "
          f"{str(ours_stats.get('wall_median_s')):>8} {str(ours_stats.get('wall_max_s')):>7}")
    print(f"{'cloudict':<10} {theirs_stats.get('replies', 0):>8} {str(theirs_stats.get('wall_mean_s')):>7} "
          f"{str(theirs_stats.get('wall_median_s')):>8} {str(theirs_stats.get('wall_max_s')):>7}")

    print(f"\nour replies split: {len(ours_searched)} searched, {len(book_replies)} book hits "
          f"(book replies are answered without a search)")
    if searched_stats.get("replies"):
        print(f"searched replies only: mean {searched_stats['wall_mean_s']} s, "
              f"max {searched_stats['wall_max_s']} s")
    overheads = [m["wall_s"] - m["search_s"] for m in ours_searched if "search_s" in m]
    if overheads:
        print(f"end-to-end minus engine-reported search time (protocol + I/O overhead): "
              f"mean {sum(overheads) / len(overheads):.3f} s, max {max(overheads):.3f} s "
              f"over {len(overheads)} searched replies")

    print(f"\n2-second budget target: {len(target_misses)} of {len(ours)} our replies exceeded "
          f"2.0 s end-to-end"
          + (f" ({len(search_target_misses)} of them also exceeded it inside the engine's own "
             f"search time)" if search_target_misses else ""))
    for m in target_misses[:8]:
        print(f"  ply {m['ply']} ({m['engine']}): wall {m['wall_s']} s, "
              f"search {m.get('search_s')} s, nodes {m.get('nodes')}")
    print(f"3-second deadline: {len(deadline_misses)} replies exceeded it "
          f"(fixtures stopped as timeout: {deadline_fixtures or 'none'})")

    worst = sorted(ours, key=lambda m: -m["wall_s"])[:5]
    print(f"\nour five slowest replies (end-to-end): "
          + ", ".join(f"{m['wall_s']} s (nodes {m.get('nodes')})" for m in worst))

engine      replies  mean s median s   max s
ours             26   0.787     0.52   2.771
cloudict         23   0.071    0.067   0.153

our replies split: 25 searched, 1 book hits (book replies are answered without a search)
searched replies only: mean 0.816 s, max 2.771 s
end-to-end minus engine-reported search time (protocol + I/O overhead): mean 0.016 s, max 0.094 s over 24 searched replies

2-second budget target: 4 of 26 our replies exceeded 2.0 s end-to-end (4 of them also exceeded it inside the engine's own search time)
  ply 3 (ours): wall 2.771 s, search 2.77 s, nodes 3412
  ply 3 (ours): wall 2.5 s, search 2.5 s, nodes 13623
  ply 5 (ours): wall 2.149 s, search 2.148 s, nodes 11493
  ply 9 (ours): wall 2.446 s, search 2.446 s, nodes 12548
3-second deadline: 0 replies exceeded it (fixtures stopped as timeout: ['win-2-seedJJ', 'win-3-seedHIKK', 'defend-2-double-threat', 'defend-3-fork-build'])

our five slowest replies (end-to-end): 2.771 s (nodes 3412), 2.5 s (nodes 13623), 2.

In [3]:
MOVE_RE = re.compile(r"^move ([A-S]{2,4})\s*$")

# protocol smoke test: identify, start as both colours, set depth, receive a move,
# return a legal move, exit.
workdir = Path(tempfile.mkdtemp(prefix="protocol_smoke_"))
env = dict(os.environ, V6_JOBS="1", V6_TIME_BUDGET="2", PYTHONDONTWRITEBYTECODE="1")
err = open(LOGS / "03_smoke_stderr.txt", "w")
t_spawn = time.perf_counter()
proc = subprocess.Popen([str(OUR_BIN)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                        stderr=err, text=True, bufsize=1, cwd=str(workdir), env=env)
lines = queue.Queue()
threading.Thread(target=lambda: [lines.put(l.rstrip("\n")) for l in proc.stdout],
                daemon=True).start()

transcript = [f"# protocol smoke test - compiled V6 ({OUR_BIN.relative_to(ROOT)})",
              f"# env: V6_JOBS=1, V6_TIME_BUDGET=2; cwd: fresh temporary directory"]


def send(cmd):
    t0 = time.perf_counter() - t_spawn
    proc.stdin.write(cmd + "\n")
    proc.stdin.flush()
    transcript.append(f"[+{t0:.3f}s] >>> {cmd}")


def collect(quiet_s=0.6, first_only=False):
    out = []
    try:
        while True:
            out.append(lines.get(timeout=quiet_s))
            if first_only:
                break
    except queue.Empty:
        pass
    t0 = time.perf_counter() - t_spawn
    for line in out:
        transcript.append(f"[+{t0:.3f}s] <<< {line}")
    return out


def read_move(timeout):
    out = []
    t0 = time.perf_counter()
    while True:
        line = lines.get(timeout=timeout)
        out.append(line)
        if MOVE_RE.match(line):
            break
    elapsed = time.perf_counter() - t0
    t0 = time.perf_counter() - t_spawn
    for line in out:
        transcript.append(f"[+{t0:.3f}s] <<< {line}")
    return MOVE_RE.match(out[-1]).group(1), out, elapsed


startup = collect(quiet_s=1.0)          # banner printed on startup
startup_first = startup[0] if startup else ""
print(f"startup: {len(startup)} banner lines; first line after "
      f"{time.perf_counter() - t_spawn:.3f}s")

send("name")
name_reply = collect(quiet_s=0.4)
identified = [l for l in name_reply if l.startswith("name ")]
print(f"identify: {identified}")

send("depth 2")
depth_reply = collect(quiet_s=0.4)
print(f"set depth: {[l for l in depth_reply if 'depth' in l.lower()]}")

send("new black")                        # start as black: receive a move (the opening)
opening, out, opening_wall = read_move(DEADLINE_S)
print(f"start as black, received a move: {opening} "
      f"(end-to-end {opening_wall:.3f}s, no search expected)")

send("new white")                        # start as white: opponent opens the center
collect(quiet_s=0.3)
send("black JJ")
collect(quiet_s=0.3)
send("next")                             # our colour after 'black JJ' toggles to white
white_move, out, white_wall = read_move(DEADLINE_S)
search_lines = [l for l in out if l.startswith(("Search Time", "Node", "Score", "Book"))]
print(f"start as white, returned a legal move: {white_move} "
      f"(end-to-end {white_wall:.3f}s)\n  stats: {search_lines}")

send("time 3")                           # probe: is a time parameter supported?
silence = collect(quiet_s=1.0)
print(f"time-parameter probe: 'time 3' produced {len(silence)} lines "
      f"(expected silence - unknown command)")

send("name")                            # engine still responsive after the ignored probe
alive = collect(quiet_s=0.4)
print(f"engine responsive after probe: {[l for l in alive if l.startswith('name ')]}")

send("exit")
t_exit = time.perf_counter()
proc.wait(timeout=10)
exit_wall = time.perf_counter() - t_exit
err.close()
shutil.rmtree(workdir, ignore_errors=True)
print(f"exit: clean exit in {exit_wall:.3f}s, returncode {proc.returncode}")

# legality of the white move on a fresh referee board: JJ (black) + our reply (white)
board = [[0] * 21 for _ in range(21)]
init_board(board)
make_move(board, msg2move("JJ"), Color.BLACK)
white = msg2move(white_move)
legal = all(board[p.x][p.y] == 0 for p in white.positions)
make_move(board, white, Color.WHITE)
no_end = check_game_end(board, white) is None
print(f"returned move {white_move} is a legal placement on the fresh board: {legal}; "
      f"game not ended by it: {no_end}")

transcript.append(f"# startup->first banner line: {startup_first!r}")
transcript.append(f"# opening reply (new black): {opening}, end-to-end {opening_wall:.3f}s")
transcript.append(f"# white reply (next): {white_move}, end-to-end {white_wall:.3f}s, "
                  f"engine-reported search lines: {search_lines}")
transcript.append(f"# legality of {white_move}: placement legal={legal}, game not ended={no_end}")
transcript.append("# 'time 3' probe: no reply - the protocol has no time command")
transcript.append(f"# exit: returncode {proc.returncode}")
transcript_path = RECORDS / "03_protocol_transcript.txt"
transcript_path.write_text("\n".join(transcript) + "\n")
print(f"\ntranscript saved to {transcript_path.relative_to(ROOT)}")
if not (legal and no_end):
    print("SMOKE TEST FAILURE RECORDED: the returned move was not legal")

startup: 16 banner lines; first line after 1.099s


identify: ['name Connect6_Butovka_Ahmed']


set depth: ['Set the search depth to 2.']
start as black, received a move: JJ (end-to-end 0.000s, no search expected)


start as white, returned a legal move: IIKK (end-to-end 0.000s)
  stats: ['Book:\tIIKK']


time-parameter probe: 'time 3' produced 0 lines (expected silence - unknown command)


engine responsive after probe: ['name Connect6_Butovka_Ahmed']
exit: clean exit in 0.009s, returncode 0
returned move IIKK is a legal placement on the fresh board: True; game not ended by it: True

transcript saved to benchmarks/assignment_coverage/records/03_protocol_transcript.txt


In [4]:
print("\n".join(transcript))
print("\n--- derived timing summary (smoke test only; sampled, not a guarantee) ---")
print(f"startup (spawn -> banner drained):        {len(startup)} lines, "
      f"first output after spawn immediately, drained within ~1.0 s quiet window")
white_was_book = any(line.startswith("Book:") for line in search_lines)
white_kind = "answered from the opening book, no search" if white_was_book else "floor-2 budget search"
print(f"opening reply (new black):              {opening_wall * 1000:.0f} ms end-to-end")
print(f"white reply (next; {white_kind}): {white_wall * 1000:.0f} ms end-to-end")
search_ms = None
for line in search_lines:
    matched = re.match(r"^Search Time:\s*([\d.]+)", line)
    if matched:
        search_ms = float(matched.group(1)) * 1000
if search_ms is not None:
    print(f"engine-reported search time for it:     {search_ms:.0f} ms "
          f"(the rest of the {white_wall * 1000:.0f} ms is protocol + I/O + startup of the move loop)")
print(f"exit latency:                             {exit_wall * 1000:.0f} ms")

print("\n--- protocol time-parameter statement ---")
print("There is no time command; the per-move wall budget is configured by the "
      "V6_TIME_BUDGET environment variable before the engine starts.")

print("\n--- GUI compatibility ---")
print("UNTESTED: this notebook exercised only the console text protocol. No GUI is present "
      "in this repository to check against, and console tests cannot establish GUI "
      "compatibility - nothing here claims it.")

# protocol smoke test - compiled V6 (dist/Connect6_Butovka_Ahmed)
# env: V6_JOBS=1, V6_TIME_BUDGET=2; cwd: fresh temporary directory
[+1.099s] <<< On help for GameEngine Connect6_Butovka_Ahmed
[+1.099s] <<<  name        - print the name of the Game Engine.
[+1.099s] <<<  print       - print the board.
[+1.099s] <<<  exit/quit   - quit the game.
[+1.099s] <<<  black XXXX  - place the black stone on the position XXXX on the board.
[+1.099s] <<<  white XXXX  - place the white stone on the position XXXX on the board, X is from A to S.
[+1.099s] <<<  next        - the engine will search the move for the next step.
[+1.099s] <<<  move XXXX   - tell the engine that the opponent made the move XXXX,
[+1.099s] <<<               and the engine will search the move for the next step.
[+1.099s] <<<  new black   - start a new game and set the engine to black player.
[+1.099s] <<<  new white   - start a new game and set it to white.
[+1.099s] <<<  depth d     - set the min-max search depth, default i

## Notes

The smoke-test transcript is saved to `records/03_protocol_transcript.txt`. Run
`01_cloudict_positions.ipynb` first; then:
`uv run python scripts/nbclient_run.py benchmarks/assignment_coverage/03_protocol_timing.ipynb`